In [1]:
import pandas as pd
import numpy as np
import re

from scipy.interpolate import interp1d
from scipy.interpolate import PchipInterpolator

from sklearn.metrics import mean_squared_error

In [2]:
df = pd.read_csv("../data/raw/dataset.csv")

ce_cols = [c for c in df.columns if c.endswith("CE")]
pe_cols = [c for c in df.columns if c.endswith("PE")]

print(df.shape)

(975, 30)


In [3]:
def get_strike(col):
    return int(
        re.search(
            r'JAN26(\d+)(CE|PE)$',
            col
        ).group(1)
    )

ce_cols = sorted(ce_cols,key=get_strike)
pe_cols = sorted(pe_cols,key=get_strike)

ce_strikes = np.array([get_strike(c) for c in ce_cols])
pe_strikes = np.array([get_strike(c) for c in pe_cols])

print(ce_strikes)
print(pe_strikes)

[25200 25300 25400 25500 25600 25700 25800 25900 26000 26100 26200 26300
 26400 26500]
[23800 23900 24000 24100 24200 24300 24400 24500 24600 24700 24800 24900
 25000 25100]


In [4]:
def linear_row(row, cols, strikes):

    values = row[cols].astype(float).values.copy()

    mask = ~np.isnan(values)

    if mask.sum() <= 1:
        return values

    f = interp1d(
        strikes[mask],
        values[mask],
        kind="linear",
        fill_value="extrapolate",
        bounds_error=False
    )

    values[~mask] = f(strikes[~mask])

    return values

In [5]:
def quadratic_row(row, cols, strikes):

    values = row[cols].astype(float).values.copy()

    mask = ~np.isnan(values)

    if mask.sum() < 3:
        return linear_row(row, cols, strikes)

    f = interp1d(
        strikes[mask],
        values[mask],
        kind="quadratic",
        fill_value="extrapolate",
        bounds_error=False
    )

    values[~mask] = f(strikes[~mask])

    return values

In [6]:
def pchip_row(row, cols, strikes):

    values = row[cols].astype(float).values.copy()

    mask = ~np.isnan(values)

    if mask.sum() <= 1:
        return values

    f = PchipInterpolator(
        strikes[mask],
        values[mask],
        extrapolate=True
    )

    values[~mask] = f(strikes[~mask])

    return values

In [7]:
def evaluate_method(method_name,
                    interpolation_function,
                    hide_fraction=0.20,
                    seed=42):

    np.random.seed(seed)

    temp = df.copy()

    known_locations = []

    for col in ce_cols + pe_cols:

        valid_rows = temp.index[
            temp[col].notna()
        ].tolist()

        for r in valid_rows:
            known_locations.append((r,col))

    known_locations = np.array(
        known_locations,
        dtype=object
    )

    n_hide = int(
        len(known_locations)
        * hide_fraction
    )

    selected_idx = np.random.choice(
        len(known_locations),
        n_hide,
        replace=False
    )

    hidden = []

    for idx in selected_idx:

        row,col = known_locations[idx]

        true_value = temp.loc[row,col]

        hidden.append(
            (row,col,true_value)
        )

        temp.loc[row,col] = np.nan

    # Fill CE
    for idx in temp.index:

        temp.loc[idx,ce_cols] = interpolation_function(
            temp.loc[idx],
            ce_cols,
            ce_strikes
        )

        temp.loc[idx,pe_cols] = interpolation_function(
            temp.loc[idx],
            pe_cols,
            pe_strikes
        )

    y_true = []
    y_pred = []

    for row,col,true_value in hidden:

        y_true.append(true_value)
        y_pred.append(temp.loc[row,col])

    mse = mean_squared_error(
        y_true,
        y_pred
    )

    return mse

In [8]:
results = []

for seed in [1,2,3,4,5]:

    linear_mse = evaluate_method(
        "linear",
        linear_row,
        seed=seed
    )

    quadratic_mse = evaluate_method(
        "quadratic",
        quadratic_row,
        seed=seed
    )

    pchip_mse = evaluate_method(
        "pchip",
        pchip_row,
        seed=seed
    )

    results.append({
        "seed":seed,
        "linear":linear_mse,
        "quadratic":quadratic_mse,
        "pchip":pchip_mse
    })

results_df = pd.DataFrame(results)

results_df

,seed,linear,quadratic,pchip
0,1,0.000066,0.000162,0.000135
1,2,0.000094,0.000460,0.000512
2,3,0.000136,0.001499,0.000978
3,4,0.000051,0.000142,0.000091
4,5,0.000103,0.000686,0.001466


In [9]:
results_df.mean(numeric_only=True)

seed         3.000000
linear       0.000090
quadratic    0.000590
pchip        0.000637
dtype: float64